In [2]:
import pandas as pd
import numpy as np
import os
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil
import pytz  # ensure this is at the top of your script

root_folder = '../../../../'
planning_data = os.path.join(root_folder,'planning')
aware_folder = os.path.join(root_folder,'3_Analytics_Warehouse_Backend')

traffic_email_path = os.path.join(planning_data,'3_emails_from_traffic')
phase2_sensor_path = os.path.join(planning_data,'4_phase_2_sensors/P2')

column_names_location = os.path.join(aware_folder,'1_inputs')#'../../../1_inputs'
parquet_intermediate_location = os.path.join(aware_folder,'2_planning_processing_files')#'../../../2_planning_processing_files'
sensors = pd.read_csv(os.path.join(planning_data,'Sensor_Mapping.csv'))

clear_folder = os.path.join(traffic_email_path, 'clear')

In [3]:
# Process all new files and combine them all.
# dfs = []
["2025-01_WTC-PED-MONTHLY-RPT.parquet","2025-02_WTC-PED-MONTHLY-RPT.parquet","2025-03_WTC-PED-MONTHLY-RPT.parquet","2025-04_WTC_MONTHLY_REPORT.parquet","2025-05_WTC-PED_MONTHLY.parquet"]
parquet_files = "2025-05_WTC-PED_MONTHLY.parquet"

# for file in parquet_files:
file_path = os.path.join(parquet_intermediate_location,"p1_sensors_parquet","processed", parquet_files)
    # arch_path = os.path.join(parquet_intermediate_location,"p1_sensors_parquet","processed",file)
    # try:
df = pd.read_parquet(file_path, engine="pyarrow")
print(f"Successfully read: {parquet_files}")

df_filtered = df\
    .drop(["To Time"], axis = 1)\
    .melt(id_vars = "From",var_name= "Location",value_name="Count")\
    .merge(sensors, left_on = "Location",right_on = "Sensor Names")\
    .query("`Hub In/Out` == 'IN'")\
    .assign(From=lambda d: pd.to_datetime(d["From"]))\
    .query("From.dt.month == 4")\
    .reset_index(drop = True)\
    .loc[
        lambda d: d["Location - Monthly Ped Report"].notna() & 
                (d["Location - Monthly Ped Report"].astype(str).str.strip() != "")
    ]

total_count = df_filtered["Count"].sum()
print("Total count for May (IN only):", total_count)


# 1️⃣ Total of all counts (no filter)
total_all = df_filtered["Count"].sum()

# 2️⃣ Total where Count < 500
total_under_500 = df_filtered.loc[df_filtered["Count"] <= 500, "Count"].sum()

# 3️⃣ Total where Count < 4000
total_under_4000 = df_filtered.loc[df_filtered["Count"] <= 4000, "Count"].sum()

print(f"Total (no filter): {total_all}")
print(f"Total (Count < 500): {total_under_500}")
print(f"Total (Count < 4000): {total_under_4000}")
df_filtered#.head()

# df.head()
df_filtered

Successfully read: 2025-05_WTC-PED_MONTHLY.parquet
Total count for May (IN only): 2811709.0
Total (no filter): 2811709.0
Total (Count < 500): 835239.0
Total (Count < 4000): 835239.0


,From,Location,Count,Sensor Number,Sensor Names,Weekly Avg Mapping,Interior Hub In Out,Monthly Peds Hub in Out,Hub In/Out,Exit Entry,Floor,Location - Monthly Ped Report,Location - Retail Report,Consolidated Location,Zone Description - Retail Report,Zone Description - Sensor Map,Zone Number,Sensor Type,Oculus Grouping
3456,2025-04-25 00:00:00,Z02_T4-LibertySt_EastEsc46_IN,1.0,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN
3457,2025-04-25 00:05:00,Z02_T4-LibertySt_EastEsc46_IN,4.0,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN
3458,2025-04-25 00:10:00,Z02_T4-LibertySt_EastEsc46_IN,7.0,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN
3459,2025-04-25 00:15:00,Z02_T4-LibertySt_EastEsc46_IN,12.0,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN
3460,2025-04-25 00:20:00,Z02_T4-LibertySt_EastEsc46_IN,8.0,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143419,2025-04-30 23:35:00,Z28_1Train-C1-SConc_AllDoors_IN,0.0,179,Z28_1Train-C1-SConc_AllDoors_IN,9.0,IN,IN,IN,Entries,C1 Balcony Level,1 Train South,NaN,1 Train South,NaN,C1 South Concourse 1 Train Entrance,28,Phase 1 Sensor,NaN
143420,2025-04-30 23:40:00,Z28_1Train-C1-SConc_AllDoors_IN,2.0,179,Z28_1Train-C1-SConc_AllDoors_IN,9.0,IN,IN,IN,Entries,C1 Balcony Level,1 Train South,NaN,1 Train South,NaN,C1 South Concourse 1 Train Entrance,28,Phase 1 Sensor,NaN
143421,2025-04-30 23:45:00,Z28_1Train-C1-SConc_AllDoors_IN,0.0,179,Z28_1Train-C1-SConc_AllDoors_IN,9.0,IN,IN,IN,Entries,C1 Balcony Level,1 Train South,NaN,1 Train South,NaN,C1 South Concourse 1 Train Entrance,28,Phase 1 Sensor,NaN
143422,2025-04-30 23:50:00,Z28_1Train-C1-SConc_AllDoors_IN,3.0,179,Z28_1Train-C1-SConc_AllDoors_IN,9.0,IN,IN,IN,Entries,C1 Balcony Level,1 Train South,NaN,1 Train South,NaN,C1 South Concourse 1 Train Entrance,28,Phase 1 Sensor,NaN


In [4]:
# Columns
cols = pd.read_csv(os.path.join(column_names_location,"columns.csv"))["Column Names"].tolist()
cols_p2 = pd.read_csv(os.path.join(column_names_location,"p2_columns.csv"))["Column Names"].tolist()
cols_veh = pd.read_csv(os.path.join(column_names_location,"veh_cols.csv"))["Column Names"].tolist()
sensors = pd.read_csv(os.path.join(planning_data,'Sensor_Mapping.csv'))
sensors

,Sensor Number,Sensor Names,Weekly Avg Mapping,Interior Hub In Out,Monthly Peds Hub in Out,Hub In/Out,Exit Entry,Floor,Location - Monthly Ped Report,Location - Retail Report,Consolidated Location,Zone Description - Retail Report,Zone Description - Sensor Map,Zone Number,Sensor Type,Oculus Grouping
0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,IN,0,IN,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
1,2,Z01_T4-ChurchSt_RevDoor_OUT,NaN,OUT,0,OUT,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
2,3,Z01_T4-ChurchSt_SwingDoor_IN,NaN,IN,0,IN,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
3,4,Z01_T4-ChurchSt_SwingDoor_OUT,NaN,OUT,0,OUT,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
4,5,Z01_T4-ChurchSt_WestEsc_UP,NaN,NaN,0,NaN,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,269,Z42_CenterPassage_Stair_OUT,NaN,OUT,NaN,OUT,NaN,C2 Main Level,NaN,Center West Passage (Level C2),Center West Passage (Level C2),Center-west Passage between Oculus and West Co...,Center-west Passage between Oculus and West Co...,42,Phase 2 Sensor,Oculus Floor
269,270,Z43_NorthWestPassage_IN,NaN,IN,NaN,IN,NaN,C2 Main Level,NaN,NaN,NaN,Northwest Passage between Oculus and West Conc...,Northwest Passage between Oculus and West Conc...,43,Phase 2 Sensor,NaN
270,271,Z43_NorthWestPassage_OUT,NaN,OUT,NaN,OUT,NaN,C2 Main Level,NaN,NaN,NaN,Northwest Passage between Oculus and West Conc...,Northwest Passage between Oculus and West Conc...,43,Phase 2 Sensor,NaN
271,272,Z43_SouthWestPassage_IN,NaN,IN,NaN,IN,NaN,C2 Main Level,NaN,NaN,NaN,Southwest Passage between Oculus and West Conc...,Northwest Passage between Oculus and West Conc...,43,Phase 2 Sensor,NaN


In [5]:

import os
import pandas as pd

parquet_files = [
    "2025-01_WTC-PED-MONTHLY-RPT.parquet",
    "2025-02_WTC-PED-MONTHLY-RPT.parquet",
    "2025-03_WTC-PED-MONTHLY-RPT.parquet",
    "2025-04_WTC_MONTHLY_REPORT.parquet",
    "2025-05_WTC-PED_MONTHLY.parquet",
]

summaries = []
month = 0
for fname in parquet_files:
    month = month + 1
    file_path = os.path.join(parquet_intermediate_location, "p1_sensors_parquet", "processed", fname)
    df = pd.read_parquet(file_path, engine="pyarrow")
    print(f"Successfully read: {fname}")

    tmp = (
        df
        .drop(["To Time"], axis=1, errors="ignore")
        .melt(id_vars="From", var_name="Location", value_name="Count")
        .merge(sensors, left_on="Location", right_on="Sensor Names", how="left")
        .query("`Monthly Peds Hub in Out` == 'IN'")
        .assign(From=lambda d: pd.to_datetime(d["From"]))\
        .query("From.dt.month == @month")
        # .loc[
        #     lambda d: d["Location - Monthly Ped Report"].notna() &
        #               (d["Location - Monthly Ped Report"].astype(str).str.strip() != "")
        # ]
         .drop_duplicates(subset=["From", "Location", "Count", "Location - Monthly Ped Report"])
    )

    # Label rows by month (YYYY-MM). If a file has multiple months, they’ll be separated correctly.
    tmp["Period"] = tmp["From"].dt.to_period("M").astype(str)

    # Build per-month totals
    grouped = tmp.groupby("Period")["Count"]
    month_summary = pd.DataFrame({
        "Total (no filter)": grouped.sum(),
        "Total (Count ≤ 500)": tmp.loc[tmp["Count"] <= 500].groupby("Period")["Count"].sum(),
        "Total (Count ≤ 4000)": tmp.loc[tmp["Count"] <= 4000].groupby("Period")["Count"].sum(),
    }).fillna(0)

    summaries.append(month_summary)

# Combine all files; if a month appears in multiple files, this sums them.
summary = (
    pd.concat(summaries)
      .groupby(level=0)
      .sum()
      .sort_index()
)

print("\nMonthly summary:")
print(summary)

# If you want the month as an index sorted chronologically already, you’re done.
# If you prefer a column:
summary_reset = summary.reset_index(names="Month")
summary_reset



Successfully read: 2025-01_WTC-PED-MONTHLY-RPT.parquet


C:\Users\schew\AppData\Local\Temp\ipykernel_1772\3046179642.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  }).fillna(0)


Successfully read: 2025-02_WTC-PED-MONTHLY-RPT.parquet


C:\Users\schew\AppData\Local\Temp\ipykernel_1772\3046179642.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  }).fillna(0)


Successfully read: 2025-03_WTC-PED-MONTHLY-RPT.parquet


C:\Users\schew\AppData\Local\Temp\ipykernel_1772\3046179642.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  }).fillna(0)


Successfully read: 2025-04_WTC_MONTHLY_REPORT.parquet
Successfully read: 2025-05_WTC-PED_MONTHLY.parquet

Monthly summary:
         Total (no filter)  Total (Count ≤ 500)  Total (Count ≤ 4000)
Period                                                               
2025-01          3824628.0            3812856.0             3817610.0
2025-02          3308857.0            3308857.0             3308857.0
2025-03          4058960.0            4057941.0             4058960.0
2025-04          4447792.0            4446223.0             4447792.0
2025-05          6137189.0            4542710.0             4546407.0


,Month,Total (no filter),Total (Count ≤ 500),Total (Count ≤ 4000)
0,2025-01,3824628.0,3812856.0,3817610.0
1,2025-02,3308857.0,3308857.0,3308857.0
2,2025-03,4058960.0,4057941.0,4058960.0
3,2025-04,4447792.0,4446223.0,4447792.0
4,2025-05,6137189.0,4542710.0,4546407.0


In [6]:

import os
import pandas as pd

parquet_files = [
    "2024-06_WTC-PED-MONTHLY-RPT.parquet",
    "2024-07_WTC-PED-MONTHLY-RPT.parquet",
    "2024-08_WTC-PED-MONTHLY-RPT.parquet",
    "2024-09_WTC-PED-MONTHLY-RPT.parquet",
    "2024-10_WTC-PED-MONTHLY-RPT.parquet",
]

summaries = []
month = 5
for fname in parquet_files:
    month = month + 1
    file_path = os.path.join(parquet_intermediate_location, "p1_sensors_parquet", "processed", fname)
    df = pd.read_parquet(file_path, engine="pyarrow")
    print(f"Successfully read: {fname}")

    tmp = (
        df
        .drop(["To Time"], axis=1, errors="ignore")
        .melt(id_vars="From", var_name="Location", value_name="Count")
        .merge(sensors, left_on="Location", right_on="Sensor Names", how="left")
        .query("`Monthly Peds Hub in Out` == 'IN'")     
        .assign(From=lambda d: pd.to_datetime(d["From"]))\
        .query("From.dt.month == @month")
        # .loc[
        #     lambda d: d["Location - Monthly Ped Report"].notna() &
        #               (d["Location - Monthly Ped Report"].astype(str).str.strip() != "")
        # ]
         .drop_duplicates(subset=["From", "Location", "Count", "Location - Monthly Ped Report"])
    )

    # Label rows by month (YYYY-MM). If a file has multiple months, they’ll be separated correctly.
    tmp["Period"] = tmp["From"].dt.to_period("M").astype(str)

    # Build per-month totals
    grouped = tmp.groupby("Period")["Count"]
    month_summary = pd.DataFrame({
        "Total (no filter)": grouped.sum(),
        "Total (Count ≤ 500)": tmp.loc[tmp["Count"] <= 500].groupby("Period")["Count"].sum(),
        "Total (Count ≤ 4000)": tmp.loc[tmp["Count"] <= 4000].groupby("Period")["Count"].sum(),
    }).fillna(0)

    summaries.append(month_summary)

# Combine all files; if a month appears in multiple files, this sums them.
summary = (
    pd.concat(summaries)
      .groupby(level=0)
      .sum()
      .sort_index()
)

print("\nMonthly summary:")
print(summary)

# If you want the month as an index sorted chronologically already, you’re done.
# If you prefer a column:
summary_reset = summary.reset_index(names="Month")
summary_reset



Successfully read: 2024-06_WTC-PED-MONTHLY-RPT.parquet
Successfully read: 2024-07_WTC-PED-MONTHLY-RPT.parquet
Successfully read: 2024-08_WTC-PED-MONTHLY-RPT.parquet
Successfully read: 2024-09_WTC-PED-MONTHLY-RPT.parquet
Successfully read: 2024-10_WTC-PED-MONTHLY-RPT.parquet

Monthly summary:
         Total (no filter)  Total (Count ≤ 500)  Total (Count ≤ 4000)
Period                                                               
2024-06          4605466.0            4605466.0             4605466.0
2024-07          4939542.0            4939035.0             4939542.0
2024-08          4732953.0            4732953.0             4732953.0
2024-09          4708430.0            4708430.0             4708430.0
2024-10          5134558.0            5134558.0             5134558.0


,Month,Total (no filter),Total (Count ≤ 500),Total (Count ≤ 4000)
0,2024-06,4605466.0,4605466.0,4605466.0
1,2024-07,4939542.0,4939035.0,4939542.0
2,2024-08,4732953.0,4732953.0,4732953.0
3,2024-09,4708430.0,4708430.0,4708430.0
4,2024-10,5134558.0,5134558.0,5134558.0


In [7]:
import pandas as pd

# Read file
file_path = os.path.join(
    parquet_intermediate_location, "p1_sensors_parquet", "processed",
    "2025-01_WTC-PED-MONTHLY-RPT.parquet"
)
df = (pd.read_parquet(file_path, engine="pyarrow")
    .drop(["To Time"], axis=1, errors="ignore")   
    .melt(id_vars="From", var_name="Location", value_name="Count")
    .merge(sensors, left_on="Location", right_on="Sensor Names", how="left")
    .query("`Monthly Peds Hub in Out` == 'IN'")
    .assign(From=lambda d: pd.to_datetime(d["From"]))\
    .query("From.dt.month == 1")
    # # .loc[
    # #     lambda d: d["Location - Monthly Ped Report"].notna() &
    # #                 (d["Location - Monthly Ped Report"].astype(str).str.strip() != "")
    # #     ]
    .drop_duplicates(subset=["From", "Location", "Count", "Location - Monthly Ped Report"])
)
# Derive date from 'From'
df["Date"] = pd.to_datetime(df["From"]).dt.date
# df
# Select all columns after the first two, coerce to numeric
df_to_group = df[["Date","Count"]]  # assumes col0='From', col1='To'
# data = df[data_cols].apply(pd.to_numeric, errors="coerce")

# 1) Per-date sum for each column (wide table)
by_date_wide = df_to_group.groupby(["Date"]).sum().sort_index()
by_date_wide
# 2) Per-date total across *all* data columns (single total per day)
# by_date_total = by_date_wide.sum(axis=1, min_count=1).to_frame(name="Total")

# print("Per-date per-column sums:")
# print(by_date_wide.head())
# data
# print("\nPer-date totals across all columns:")
# print(by_date_total.head())
# by_date_total


,Count
Date,
2025-01-01,107303.0
2025-01-02,166850.0
2025-01-03,145758.0
2025-01-04,110401.0
2025-01-05,78943.0
2025-01-06,139359.0
2025-01-07,162050.0
2025-01-08,168273.0
2025-01-09,146088.0


In [8]:
df

,From,Location,Count,Sensor Number,Sensor Names,Weekly Avg Mapping,Interior Hub In Out,Monthly Peds Hub in Out,Hub In/Out,Exit Entry,Floor,Location - Monthly Ped Report,Location - Retail Report,Consolidated Location,Zone Description - Retail Report,Zone Description - Sensor Map,Zone Number,Sensor Type,Oculus Grouping,Date
44352,2025-01-01 00:00:00,Z02_T4-LibertySt_EastEsc46_IN,0,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN,2025-01-01
44353,2025-01-01 00:05:00,Z02_T4-LibertySt_EastEsc46_IN,8,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN,2025-01-01
44354,2025-01-01 00:10:00,Z02_T4-LibertySt_EastEsc46_IN,6,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN,2025-01-01
44355,2025-01-01 00:15:00,Z02_T4-LibertySt_EastEsc46_IN,3,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN,2025-01-01
44356,2025-01-01 00:20:00,Z02_T4-LibertySt_EastEsc46_IN,2,9,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,IN,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,4 WTC Transit Lobby,NaN,T4 Transit Lobby Entrance from Liberty St,2,Phase 1 Sensor,NaN,2025-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1928731,2025-01-31 23:35:00,Z31_T3Elev-C2_OUTOFELEV_IN,2,189,Z31_T3Elev-C2_OUTOFELEV_IN,NaN,IN,IN,IN,NaN,NaN,NaN,NaN,NaN,NaN,0,31,Phase 1 Sensor,NaN,2025-01-31
1928732,2025-01-31 23:40:00,Z31_T3Elev-C2_OUTOFELEV_IN,0,189,Z31_T3Elev-C2_OUTOFELEV_IN,NaN,IN,IN,IN,NaN,NaN,NaN,NaN,NaN,NaN,0,31,Phase 1 Sensor,NaN,2025-01-31
1928733,2025-01-31 23:45:00,Z31_T3Elev-C2_OUTOFELEV_IN,0,189,Z31_T3Elev-C2_OUTOFELEV_IN,NaN,IN,IN,IN,NaN,NaN,NaN,NaN,NaN,NaN,0,31,Phase 1 Sensor,NaN,2025-01-31
1928734,2025-01-31 23:50:00,Z31_T3Elev-C2_OUTOFELEV_IN,0,189,Z31_T3Elev-C2_OUTOFELEV_IN,NaN,IN,IN,IN,NaN,NaN,NaN,NaN,NaN,NaN,0,31,Phase 1 Sensor,NaN,2025-01-31


In [9]:
database_location = os.path.join(aware_folder,'5_database')#'../../../5_database'
cols = pd.read_csv(os.path.join(column_names_location,"columns.csv"))["Column Names"].tolist()

df_database_p1 = pd.read_parquet(os.path.join(database_location,'database_cleaned_p1.parquet'),columns= cols, engine="pyarrow")
df_filtered = df_database_p1\
    .query("From.dt.month == 5")\
    .query("From.dt.year == 2025")\
    .drop(["To Time"], axis = 1)\
    .melt(id_vars = "From",var_name= "Location",value_name="Count")\
    .merge(sensors, left_on = "Location",right_on = "Sensor Names")\
    .query("`Hub In/Out` == 'IN'")\
    .assign(From=lambda d: pd.to_datetime(d["From"]))\
    .reset_index(drop = True)\
    .loc[
        lambda d: d["Location - Monthly Ped Report"].notna() & 
                (d["Location - Monthly Ped Report"].astype(str).str.strip() != "")\
            
    ]
df_filtered["Period"] = df_filtered["From"].dt.to_period("D").astype(str)

df_filtered= df_filtered.groupby("Period")["Count"].count()
df_filtered

Period
2025-05-01    16416
2025-05-02    16416
2025-05-03    16416
2025-05-04    16416
2025-05-05    16416
2025-05-06    16412
2025-05-07    16416
2025-05-08    16416
2025-05-09    16416
2025-05-10    16416
2025-05-11    16416
2025-05-12    16416
2025-05-13    16413
2025-05-14    16416
2025-05-15    16416
2025-05-16    16416
2025-05-17    16416
2025-05-18    16416
2025-05-19    16416
2025-05-20    16414
2025-05-21    16416
2025-05-22    16416
2025-05-23    16416
2025-05-24    16416
2025-05-25    16415
2025-05-26    16416
2025-05-27    16416
2025-05-28    16414
2025-05-29    16416
2025-05-30    16416
2025-05-31    16416
Name: Count, dtype: int64

In [22]:
df_database_check = df_database_p1
df_database_check["Period"] = df_database_check["From"].dt.to_period("D").astype(str)

df_database_check = df_database_check.groupby("Period").count()    #.melt(id_vars = "From",var_name= "Location",value_name="Count")\


df_database_check

,From,To Time,Z01_T4-ChurchSt_RevDoor_IN,Z01_T4-ChurchSt_RevDoor_OUT,Z01_T4-ChurchSt_SwingDoor_IN,Z01_T4-ChurchSt_SwingDoor_OUT,Z02_T4-LibertySt_EastEsc46_IN,Z02_T4-LibertySt_EastEsc46_OUT,Z02_T4-LibertySt_WestEsc45_IN,Z02_T4-LibertySt_WestEsc45_OUT,...,Z27_T3TransitLobby_Stair50_DOWN_IN,Z27_T3TransitLobby_Stair50_UP_OUT,Z28_T3Elev-C1_OUTOFELEV_IN,Z28_T3Elev-C1_INTOELEV_OUT,Z28_1Train-C1-SConc_AllDoors_IN,Z28_1Train-C1-SConc_AllDoors_OUT,Z31_T3Elev-C2_OUTOFELEV_IN,Z31_T3Elev-C2_INTOELEV_OUT,Z29_C2_SWOculus_HubElev_OUTOFELEV,Z29_C2_SWOculus_HubElev_INTOELEV
Period,,,,,,,,,,,,,,,,,,,,,
2020-02-24,288,288,288,288,288,288,288,288,288,288,...,288,288,288,288,288,288,288,288,288,288
2020-02-25,288,288,288,288,288,288,288,288,288,288,...,288,288,288,288,288,288,288,288,288,288
2020-02-26,288,288,288,288,288,288,288,288,288,288,...,288,288,288,288,288,288,288,288,288,288
2020-02-27,288,288,288,288,288,288,288,288,288,288,...,288,288,288,288,288,288,288,288,288,288
2020-02-28,288,288,288,288,288,288,288,288,288,288,...,288,288,288,288,288,288,288,288,288,288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-15,288,288,288,288,288,288,288,288,288,288,...,288,288,288,288,288,288,288,288,288,288
2025-11-16,288,288,288,288,288,288,288,288,288,288,...,288,288,288,288,288,288,288,288,288,288
2025-11-17,288,288,288,288,288,288,288,288,288,288,...,288,288,288,288,288,288,288,288,288,288
